# [1] Imports & Setup (Dual Run Modes)


In [1]:
# ─────────────────────────────────────────────────────────────────────────────
# [1] Setup & Imports
# ─────────────────────────────────────────────────────────────────────────────
import os
import sys
import glob
import json
import shutil
from pathlib import Path
from datetime import datetime
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from scipy.io import savemat

# Repo root on path
repo_root = Path.cwd().resolve()
while repo_root.name and repo_root.name != "EveryMotor":
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

print(f"Repository Root: {repo_root}")


Repository Root: D:\KangDH\EveryMotor

# [2] 모터 모델 및 실행 모드 설정


In [1]:
# ─────────────────────────────────────────────────────────────────────────────
# [2] Configuration: Model Scale & Run Mode Selection
# ─────────────────────────────────────────────────────────────────────────────
# Define motor file paths for each local/remote configuration
# (Update these paths to match your local folders on different machines)
MOTOR_FILES = {
    'Ref': r"D:/KangDH/Thesis/e10/refModel/e10Turn6V261.mot",
    'HalfSC': r"D:/KangDH/Thesis/e10/SLFEA_Half/e10Turn6V261SLFEA_Half.mot",
    'SC': r"D:/KangDH/Thesis/e10/SLFEA/e10Turn6V261SLFE.mot"
}

out_dir = Path("map_exports")
out_dir.mkdir(parents=True, exist_ok=True)

# ── 1. Path existence check for all Model Scales ─────────────────────────────
print("=== [Data & Model Path Existence Check] ===")
for scale in ['Ref', 'HalfSC', 'SC']:
    mot_p = MOTOR_FILES.get(scale)
    json_p = out_dir / f"JEET_ACLoss_{scale}_Map_Summary.json"
    
    mot_exists = Path(mot_p).exists() if mot_p else False
    json_exists = json_p.exists()
    
    print(f"[{scale}]")
    print(f"  - .mot file path: {mot_p}")
    print(f"    -> Exists? {'[YES]' if mot_exists else '[NO]'}")
    print(f"  - JSON summary:  {json_p}")
    print(f"    -> Exists? {'[YES]' if json_exists else '[NO]'}")
print("===========================================\n")

MODEL_SCALE = 'HalfSC'  # Options: 'Ref' (k_Radial=1.0), 'HalfSC' (k_Radial=1.5), 'SC' (k_Radial=2.0)
RUN_FEA_SWEEP = False    # True: Run FEA inside Motor-CAD. False: Bypasses connection, loads JSON map only.

mot_file_path = MOTOR_FILES.get(MODEL_SCALE)
print(f"Selected Model Scale: {MODEL_SCALE}")
print(f"Target Motor File: {mot_file_path}")
print(f"FEA Sweep Execution: {'ENABLED' if RUN_FEA_SWEEP else 'DISABLED (Offline Mode)'}")

# Output summary paths
json_summary_path = out_dir / f"JEET_ACLoss_{MODEL_SCALE}_Map_Summary.json"
mat_summary_path = out_dir / f"JEET_ACLoss_{MODEL_SCALE}_Map_Summary.mat"
rbf_model_path = out_dir / f"AF_RBF_model_{MODEL_SCALE}.json"

# Convert to SI units (will be updated dynamically if connected, else uses fallbacks)
COND_WIDTH_MM, COND_HEIGHT_MM, ACTIVE_LENGTH_MM = 2.5, 2.5, 150.0
n_parallel, n_turns = 1, 6

mcad = None
if RUN_FEA_SWEEP:
    print("Connecting to Motor-CAD instance...")
    try:
        import ansys.motorcad.core as pymotorcad
        mcad = pymotorcad.MotorCAD(open_new_instance=False)
        if Path(mot_file_path).exists():
            print(f"Loading selected model: {mot_file_path}")
            mcad.load_from_file(mot_file_path)
        else:
            print(f"[WARN] Motor-CAD file not found at: {mot_file_path}")
            print("Will attempt to proceed with the currently active Motor-CAD model.")
        COND_WIDTH_MM = float(mcad.get_variable("Copper_Width"))
        COND_HEIGHT_MM = float(mcad.get_variable("Copper_Height"))
        ACTIVE_LENGTH_MM = float(mcad.get_variable("Stator_Lam_Length"))
        n_parallel = int(mcad.get_variable("ParallelPaths"))
        n_turns = int(mcad.get_variable("MagTurnsConductor"))
        print(f"✓ Read parameters: Width={COND_WIDTH_MM:.2f}mm, Height={COND_HEIGHT_MM:.2f}mm, Length={ACTIVE_LENGTH_MM:.1f}mm")
    except Exception as e:
        print(f"[WARN] Motor-CAD connection/read failed: {e}")
else:
    print("[Offline Mode] Motor-CAD connection bypassed.")
    print(f"Using default fallback parameters: Width={COND_WIDTH_MM}mm, Height={COND_HEIGHT_MM}mm, Length={ACTIVE_LENGTH_MM}mm")

b_m = COND_WIDTH_MM * 1e-3
h_m = COND_HEIGHT_MM * 1e-3
L_a = ACTIVE_LENGTH_MM * 1e-3

# Electrical frequency helper
POLE_PAIRS = 4
def speed_to_fe(speed_rpm, pole_pairs=POLE_PAIRS):
    return pole_pairs * speed_rpm / 60.0


=== [Data & Model Path Existence Check] ===
[Ref]
  - .mot file path: D:/KangDH/Thesis/e10/refModel/e10Turn6V261.mot
    -> Exists? [YES]
  - JSON summary:  map_exports\JEET_ACLoss_Ref_Map_Summary.json
    -> Exists? [NO]
[HalfSC]
  - .mot file path: D:/KangDH/Thesis/e10/SLFEA_Half/e10Turn6V261SLFEA_Half.mot
    -> Exists? [YES]
  - JSON summary:  map_exports\JEET_ACLoss_HalfSC_Map_Summary.json
    -> Exists? [YES]
[SC]
  - .mot file path: D:/KangDH/Thesis/e10/SLFEA/e10Turn6V261SLFE.mot
    -> Exists? [NO]
  - JSON summary:  map_exports\JEET_ACLoss_SC_Map_Summary.json
    -> Exists? [NO]

Selected Model Scale: HalfSC
Target Motor File: D:/KangDH/Thesis/e10/SLFEA_Half/e10Turn6V261SLFEA_Half.mot
FEA Sweep Execution: DISABLED (Offline Mode)
[Offline Mode] Motor-CAD connection bypassed.
Using default fallback parameters: Width=2.5mm, Height=2.5mm, Length=150.0mm

# [3] 통합 FEA Sweep 및 중복 방지 Resume 로직


In [1]:
# 이 셀은 오프라인 모드와 온라인 모드 모두에서 실행 가능한 헬퍼 기능들을 정의합니다.
SIGMA_CU = 5.8e7
MU_0 = 4.0 * np.pi * 1e-7


# [3.1] 통합 Sweep Loop 실행


In [1]:
# ─────────────────────────────────────────────────────────────────────────────
# [3] Robust FEA Sweep with Automatic Resume/Skip Logic
# ─────────────────────────────────────────────────────────────────────────────
if RUN_FEA_SWEEP:
    from tools.motorCAD.pyMCAD import mcad_default_export_dir, calc_dc_loss_kw
    
    # ── 1. Define Sweep Range ─────────────────────────────────────────────────
    CURRENT_GRID = np.linspace(0.1, 460.0, 5)   # [0.1, 115.1, 230.1, 345.1, 460.0] A
    PHASE_GRID = np.linspace(0.0, 90.0, 6)      # [0.0, 18.0, 36.0, 54.0, 72.0, 90.0] deg
    PROXIMITY_MODELS = [1, 3]                  # 1: Hybrid, 3: FullFEA
    
    FIRST_STEP = 1
    EXPORT_COLUMNS = "RegCode,Bx,By,A,J,Je,Hx,Hy,Mur"
    
    # Assemble unified sweep schedule (Speeds: 2k, 4k, 16k get full grid, 8k gets 16-point subgrid)
    sweep_schedule = []
    for prox_model in PROXIMITY_MODELS:
        for speed in [2000, 4000, 8000, 16000]:
            if speed == 8000:
                curr_list = CURRENT_GRID[1:]          # [115.1, 230.1, 345.1, 460.0] A
                phase_list = PHASE_GRID[[0, 1, 3, 5]]  # [0.0, 18.0, 54.0, 90.0] deg
            else:
                curr_list = CURRENT_GRID
                phase_list = PHASE_GRID
                
            for current in curr_list:
                for phase in phase_list:
                    sweep_schedule.append({
                        "proximity_model": prox_model,
                        "speed": speed,
                        "current": current,
                        "phase": phase
                    })
                    
    print(f"Unified sweep schedule compiled. Total points to verify: {len(sweep_schedule)}")
    
    # ── 2. Load Existing Progress to Avoid Duplicate Sweeps ────────────────────
    sweep_results = []
    if json_summary_path.exists():
        try:
            with open(json_summary_path, "r", encoding="utf-8") as f:
                sweep_results = json.load(f)
            print(f"✓ Loaded existing summary file: {json_summary_path}")
            print(f"  Existing records: {len(sweep_results)} points.")
        except Exception as e:
            print(f"[WARN] Failed to load existing JSON: {e}. Starting fresh.")
            sweep_results = []
            
    def is_point_existing(p_model, spd, curr, ph):
        for record in sweep_results:
            if (record["proximity_model"] == p_model and 
                record["speed"] == spd and 
                np.isclose(record["current"], curr, atol=1e-2) and 
                np.isclose(record["phase"], ph, atol=1e-2)):
                return True
        return False
        
    # Setup folders
    out_root = Path(mcad_default_export_dir(mcad))
    backup_root = out_root / f"ACLossCalcExport_{MODEL_SCALE}"
    backup_root.mkdir(parents=True, exist_ok=True)
    
    # Winding resistances
    try:
        R_total = float(mcad.get_variable("Resistance_MotorLAB")) * 4.0
        R_end = float(mcad.get_variable("EndWindingResistance_Lab")) * 4.0
        R_active = R_total - R_end
    except Exception as e:
        R_total, R_end, R_active = 0.0, 0.0, 0.0
        print(f"[WARN] Failed to read winding resistances: {e}")
        
    # Helper to find latest solved results folder
    def find_latest_mes(mcad_inst):
        import glob
        mcad_dir = mcad_default_export_dir(mcad_inst)
        candidates = sorted(glob.glob(os.path.join(mcad_dir, "*.mes")))
        if not candidates:
            raise FileNotFoundError(f"No .mes files found in {mcad_dir}")
        return Path(candidates[-1])

    # ── 3. FEA Sweep Loop ─────────────────────────────────────────────────────
    new_points_run = 0
    for idx, pt in enumerate(sweep_schedule):
        prox_model = pt["proximity_model"]
        speed = pt["speed"]
        current = pt["current"]
        phase = pt["phase"]
        mode_label = "Hybrid" if prox_model == 1 else "FullFEA"
        
        # Check if already present
        if is_point_existing(prox_model, speed, current, phase):
            print(f"[{idx+1}/{len(sweep_schedule)}] [Skipped] {mode_label} | Speed: {speed} RPM | I: {current:.1f} A | Phase: {phase:.1f} deg")
            continue
            
        print(f"[{idx+1}/{len(sweep_schedule)}] [Running] {mode_label} | Speed: {speed} RPM | I: {current:.1f} A | Phase: {phase:.1f} deg")
        
        # Connect & configure variables
        mcad.set_variable("ProximityLossModel", prox_model)
        mcad.set_variable("ShaftSpeed", speed)
        mcad.set_variable("RMSCurrent", current)
        mcad.set_variable("PhaseAdvance", phase)
        
        # Execute FEA
        mcad.do_magnetic_calculation()
        torque_points = int(mcad.get_variable("TorquePointsPerCycle"))
        
        # Locate results
        try:
            latest_mes = find_latest_mes(mcad)
            active_results_dir = latest_mes.parent
        except Exception as e:
            print(f"  [ERROR] Results directory not found: {e}")
            continue
            
        # Destination directory
        point_folder_name = f"{mode_label}_Speed_{speed}RPM_{current:.1f}A_{phase:.1f}deg"
        dest_point_dir = backup_root / point_folder_name
        dest_results_dir = dest_point_dir / "FEResultsData"
        
        if dest_results_dir.exists():
            shutil.rmtree(dest_results_dir)
        shutil.copytree(active_results_dir, dest_results_dir)
        
        # Export B-field TXT
        txt_path = dest_point_dir / "FEA_data.txt"
        mcad.save_fea_data(str(txt_path), FIRST_STEP, torque_points, EXPORT_COLUMNS, "", ",")
        
        # Get losses
        point_data = {
            "proximity_model": prox_model,
            "mode": mode_label,
            "speed": speed,
            "current": current,
            "phase": phase,
            "backup_dir": str(dest_point_dir)
        }
        
        if prox_model == 1:
            point_data["hybrid_total_kW"] = float(mcad.get_variable("ACLoss_Hybrid_Total"))
            point_data["hybrid_prox_kW"]  = float(mcad.get_variable("ACLoss_Hybrid_Prox_Total"))
            point_data["hybrid_skin_kW"]  = float(mcad.get_variable("ACLoss_Hybrid_SkinEffect_Total"))
        else:
            raw_pt_str = mcad.get_variable("ACLoss_FEA_OnLoad_PerTurn")
            point_data["fea_per_turn_raw"] = raw_pt_str
            point_data["fea_total_ac_kW"]  = float(mcad.get_variable("ACLoss_FEA_OnLoad_Total")) / 1000.0
            
            try:
                losses_raw = [float(x) for x in raw_pt_str.split(",") if x.strip()]
                loss_per_turn = np.array(losses_raw)
            except Exception:
                loss_per_turn = np.zeros(n_turns * n_parallel)
                
            R_dc_act_per_turn = R_active / float(n_turns)
            dc_loss_act = calc_dc_loss_kw(R_dc_act_per_turn, current)
            
            loss_active_only = loss_per_turn.copy()
            for t in range(n_turns):
                for p in range(n_parallel):
                    t_idx = t * n_parallel + p
                    if loss_active_only[t_idx] > dc_loss_act:
                        loss_active_only[t_idx] -= dc_loss_act
                    else:
                        loss_active_only[t_idx] = 0.0
                        
            ts_ac_active_only = np.sum(loss_active_only)
            point_data["ts_ac_active_only_kW"] = ts_ac_active_only
            point_data["ts_dc_active_only_kW"] = dc_loss_act * n_turns * n_parallel
            
        sweep_results.append(point_data)
        new_points_run += 1
        
        # Save JSON
        with open(json_summary_path, "w", encoding="utf-8") as f:
            json.dump(sweep_results, f, ensure_ascii=False, indent=2)
            
    if new_points_run > 0:
        mat_data = {"sweep_results": sweep_results}
        savemat(str(mat_summary_path), mat_data, do_compression=True)
        print(f"\n✓ Sweep Complete: JSON saved to {json_summary_path}, MAT saved to {mat_summary_path}")
    else:
        print("\nAll scheduled points already present. No new sweeps executed.")
else:
    print("[Offline Mode] Bypassing FEA Sweep calculation loop.")


[Offline Mode] Bypassing FEA Sweep calculation loop.

# [3c] 비활성화된 보완 스윕 셀 (Cell 7로 통합)


In [1]:
# 이 셀은 통합 버전 스케줄러(Cell 7)로 합병되어 사용하지 않습니다.
print("통합 스케줄러 Cell 7이 사용됩니다.")


통합 스케줄러 Cell 7이 사용됩니다.

# [4] id-iq 평면 AC Active Only 손실 Surface 플롯 (속도별)

ProximityLossModel = 1(Hybrid) 및 3(FullFEA/TS) 각각에 대해 속도별로 $I_d, I_q$ 평면에서의 AC Active Only 손실 Surface 플롯을 시각화합니다.

In [1]:
# ─────────────────────────────────────────────────────────────────────────────
# [4] id-iq 평면 AC Active Only 손실 Surface 플롯 (대화형 비교)
# ─────────────────────────────────────────────────────────────────────────────
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
import matplotlib.patches as mpatches

if json_summary_path.exists():
    with open(json_summary_path, "r", encoding="utf-8") as f:
        sweep_results = json.load(f)
    
    hybrid_data = [p for p in sweep_results if p["proximity_model"] == 1]
    ts_data = [p for p in sweep_results if p["proximity_model"] == 3]
    
    def process_pts(pts, is_hybrid):
        speeds = np.array([p["speed"] for p in pts])
        currents = np.array([p["current"] for p in pts])
        phases = np.array([p["phase"] for p in pts])
        
        amplitude = currents * np.sqrt(2)
        phase_rad = (phases + 90) * np.pi / 180.0
        id_vals = amplitude * np.cos(phase_rad)
        iq_vals = amplitude * np.sin(phase_rad)
        
        if is_hybrid:
            losses = np.array([p["hybrid_total_kW"] for p in pts])
        else:
            losses = np.array([p["ts_ac_active_only_kW"] for p in pts])
            
        return speeds, id_vals, iq_vals, losses, pts

    speed_colors = {2000: 'cyan', 4000: 'limegreen', 8000: 'orange', 16000: 'tomato'}
    default_colors = ['cyan', 'limegreen', 'orange', 'tomato']
    
    def create_interactive_comparison_plot(pts_hybrid, pts_ts):
        speeds_h, id_h, iq_h, losses_h, raw_h = process_pts(pts_hybrid, is_hybrid=True)
        speeds_f, id_f, iq_f, losses_f, raw_f = process_pts(pts_ts, is_hybrid=False)
        
        currents_h = np.array([p["current"] for p in raw_h])
        phases_h = np.array([p["phase"] for p in raw_h])
        currents_f = np.array([p["current"] for p in raw_f])
        phases_f = np.array([p['phase'] for p in raw_f])
        
        unique_speeds = sorted(list(set(speeds_h)))
        
        fig = plt.figure(figsize=(18, 5.5))
        fig.suptitle(f"AC Loss Comparison Map ({MODEL_SCALE}): Hybrid vs FullFEA", fontsize=13, fontweight='bold')
        
        ax_left = fig.add_subplot(131, projection='3d')
        ax_left.set_title("Hybrid (ProximityLossModel = 1)", fontsize=11, fontweight='bold')
        ax_mid = fig.add_subplot(132, projection='3d')
        ax_mid.set_title("FullFEA (ProximityLossModel = 3)", fontsize=11, fontweight='bold')
        
        legend_patches_h = []
        legend_patches_f = []
        
        for i, spd in enumerate(unique_speeds):
            color = speed_colors.get(spd, default_colors[i % len(default_colors)])
            idx_h = (speeds_h == spd)
            if np.any(idx_h) and np.sum(idx_h) >= 3:
                ax_left.plot_trisurf(id_h[idx_h], iq_h[idx_h], losses_h[idx_h], color=color, edgecolor='none', alpha=0.35)
                legend_patches_h.append(mpatches.Patch(color=color, alpha=0.35, label=f"{spd} RPM"))
            idx_f = (speeds_f == spd)
            if np.any(idx_f) and np.sum(idx_f) >= 3:
                ax_mid.plot_trisurf(id_f[idx_f], iq_f[idx_f], losses_f[idx_f], color=color, edgecolor='none', alpha=0.35)
                legend_patches_f.append(mpatches.Patch(color=color, alpha=0.35, label=f"{spd} RPM"))
                
        sc_h = ax_left.scatter(id_h, iq_h, losses_h, c='grey', s=25, picker=True, pickradius=5, edgecolors='black', alpha=0.6)
        sc_f = ax_mid.scatter(id_f, iq_f, losses_f, c='grey', s=25, picker=True, pickradius=5, edgecolors='black', alpha=0.6)
        scatters = [sc_h, sc_f]
        
        for ax, lp in [(ax_left, legend_patches_h), (ax_mid, legend_patches_f)]:

            ax.set_xlabel("I_d [A]", fontsize=8, labelpad=7)
            ax.set_ylabel("I_q [A]", fontsize=8, labelpad=7)
            ax.set_zlabel("AC Loss [kW]", fontsize=8, labelpad=7)
            ax.legend(handles=lp, fontsize=8, loc="upper right")
            
        ax_right = fig.add_subplot(133)
        ax_right.text(0.5, 0.5, "3D 플롯에서 임의의 점을 클릭한 후\nSpacebar를 누르면 속도별 비교 곡선이 출력됩니다.", 
                     ha="center", va="center", fontsize=10, color="gray")
        ax_right.set_xlabel("Speed [RPM]", fontsize=9)
        ax_right.set_ylabel("AC Loss [kW]", fontsize=9)
        ax_right.grid(True, linestyle="--", alpha=0.5)
        
        selected_pt = {"current": None, "phase": None, "id": None, "iq": None}
        highlights_h = []
        highlights_f = []
        
        annotation_h = ax_left.text2D(0.02, 0.95, "", transform=ax_left.transAxes, 
                                      bbox=dict(boxstyle="round", fc="w", alpha=0.8), fontsize=8)
        annotation_f = ax_mid.text2D(0.02, 0.95, "", transform=ax_mid.transAxes, 
                                     bbox=dict(boxstyle="round", fc="w", alpha=0.8), fontsize=8)
        annotation_h.set_visible(False)
        annotation_f.set_visible(False)
        
        def on_pick(event):
            if event.artist not in scatters:
                return
            idx = event.ind[0]
            if event.artist == sc_h:
                curr, ph = raw_h[idx]["current"], raw_h[idx]["phase"]
            else:
                curr, ph = raw_f[idx]["current"], raw_f[idx]["phase"]
            selected_pt["current"] = curr
            selected_pt["phase"] = ph
            amp = curr * np.sqrt(2)
            phase_rad = (ph + 90) * np.pi / 180.0
            selected_pt["id"] = amp * np.cos(phase_rad)
            selected_pt["iq"] = amp * np.sin(phase_rad)
            for h in highlights_h + highlights_f:
                h.remove()
            highlights_h.clear()
            highlights_f.clear()
            same_h_idx = (currents_h == curr) & (phases_h == ph)
            hh = ax_left.scatter(id_h[same_h_idx], iq_h[same_h_idx], losses_h[same_h_idx], 
                                 color='red', s=70, edgecolors='black', linewidths=1.8, zorder=10)
            highlights_h.append(hh)
            same_f_idx = (currents_f == curr) & (phases_f == ph)
            hf = ax_mid.scatter(id_f[same_f_idx], iq_f[same_f_idx], losses_f[same_f_idx], 

                                color='red', s=70, edgecolors='black', linewidths=1.8, zorder=10)
            highlights_f.append(hf)
            msg = (f"Selected: I_rms={curr:.1f}A, Phase={ph:.1f}°\n"

                   f"Id={selected_pt['id']:.1f}A, Iq={selected_pt['iq']:.1f}A\n→ Press 'Space'")
            for annot in [annotation_h, annotation_f]:
                annot.set_text(msg)
                annot.set_visible(True)
            fig.canvas.draw_idle()
            
        def on_key(event):
            if event.key != ' ' or selected_pt["current"] is None:
                return
            ax_right.clear()
            curr, ph = selected_pt["current"], selected_pt["phase"]
            curve_speeds, curve_losses_h, curve_losses_f = [], [], []
            for spd in unique_speeds:
                match_h = [p for p in raw_h if p["speed"] == spd and np.isclose(p["current"], curr) and np.isclose(p["phase"], ph)]
                match_f = [p for p in raw_f if p["speed"] == spd and np.isclose(p["current"], curr) and np.isclose(p["phase"], ph)]
                if match_h and match_f:
                    curve_speeds.append(spd)
                    curve_losses_h.append(match_h[0]["hybrid_total_kW"])
                    curve_losses_f.append(match_f[0]["ts_ac_active_only_kW"])
            ax_right.plot(curve_speeds, curve_losses_h, marker='o', linestyle='-', color='dodgerblue', linewidth=2, label="Hybrid AC Total")
            ax_right.plot(curve_speeds, curve_losses_f, marker='*', linestyle='--', color='crimson', linewidth=2, label="FullFEA AC Active Only")
            for xs, yh, yf in zip(curve_speeds, curve_losses_h, curve_losses_f):
                ax_right.annotate(f"{yh:.2f}", xy=(xs, yh), xytext=(4, 4), textcoords="offset points", fontsize=8, color="dodgerblue")
                ax_right.annotate(f"{yf:.2f}", xy=(xs, yf), xytext=(4, -12), textcoords="offset points", fontsize=8, color="crimson")
            ax_right.set_title(f"AC Loss vs Speed\n(I_rms={curr:.1f}A, Phase={ph:.1f}°)", fontsize=11, fontweight='bold')
            ax_right.set_xlabel("Speed [RPM]", fontsize=9)
            ax_right.set_ylabel("AC Loss [kW]", fontsize=9)
            ax_right.grid(True, linestyle="--", alpha=0.5)
            ax_right.legend(fontsize=9, loc="upper left")
            fig.canvas.draw_idle()
            
        fig.canvas.mpl_connect('pick_event', on_pick)
        fig.canvas.mpl_connect('key_press_event', on_key)
        plt.tight_layout()
        plt.show()
        
    if len(hybrid_data) > 0 and len(ts_data) > 0:
        create_interactive_comparison_plot(hybrid_data, ts_data)
    else:
        print("Error: Need both Hybrid and FullFEA data.")
else:
    print("JSON summary map file not found. Bypassing map surface plotting cell.")


<string>:160: UserWarning: Glyph 54540 (\N{HANGUL SYLLABLE PEUL}) missing from current font.
<string>:160: UserWarning: Glyph 47215 (\N{HANGUL SYLLABLE ROS}) missing from current font.
<string>:160: UserWarning: Glyph 50640 (\N{HANGUL SYLLABLE E}) missing from current font.
<string>:160: UserWarning: Glyph 49436 (\N{HANGUL SYLLABLE SEO}) missing from current font.
<string>:160: UserWarning: Glyph 51076 (\N{HANGUL SYLLABLE IM}) missing from current font.
<string>:160: UserWarning: Glyph 51032 (\N{HANGUL SYLLABLE YI}) missing from current font.
<string>:160: UserWarning: Glyph 51216 (\N{HANGUL SYLLABLE JEOM}) missing from current font.
<string>:160: UserWarning: Glyph 51012 (\N{HANGUL SYLLABLE EUL}) missing from current font.
<string>:160: UserWarning: Glyph 53364 (\N{HANGUL SYLLABLE KEUL}) missing from current font.
<string>:160: UserWarning: Glyph 47533 (\N{HANGUL SYLLABLE RIG}) missing from current font.
<string>:160: UserWarning: Glyph 54620 (\N{HANGUL SYLLABLE HAN}) missing from cur

# [5] Adjustment Factor (AF) 모델링

`AF = FullFEA_AC_active_only / Hybrid_AC_total`을 **(speed, Irms, phase)** 기반으로 모델링합니다.

**AF에 영향을 주는 물리적 인자:**
- **Skin effect**: 고속(고주파수) 영역에서 주파수 자승 비선형성이 꺾이는 와전류 차폐(Back-reaction) 경향을 보정합니다.
- **Proximity effect**: 고정자 전류 크기($I_{rms}$) 및 고정자 전류와 회전자 극 간의 상대 위상각($\theta$)에 의한 AC 손실 비선형 곡면을 피팅합니다.
- **Id-Iq 결합**:MTPA 운전 궤적 및 field-weakening 영역에서의 AC 손실 거동을 전류 크기와 위상각을 통해 정합시킵니다.


In [1]:
# ─────────────────────────────────────────────────────────────────────────────
# [5] Adjustment Factor (AF) 데이터 로드 및 정렬 (RBF 입력용)
# ─────────────────────────────────────────────────────────────────────────────
if json_summary_path.exists():
    print(f"[데이터 로드] {json_summary_path}")
    with open(json_summary_path, "r", encoding="utf-8") as f:
        sweep_results = json.load(f)
        
    # 백업 폴더 매칭 검증
    model_keywords = {
        'Ref': ['refModel', 'ref'],
        'HalfSC': ['SLFEA_Half', 'HalfSC'],
        'SC': ['SLFEA', 'SC']
    }
    kws = model_keywords.get(MODEL_SCALE, [])
    _non_matching = [p for p in sweep_results if "backup_dir" in p and not any(kw in p["backup_dir"] for kw in kws)]
    if _non_matching:
        print(f"  [WARNING] {len(_non_matching)}개 포인트가 {MODEL_SCALE} 모형이 아닐 수 있습니다!")
        print(f"  예: {_non_matching[0]['backup_dir']}")
    else:
        print(f"  [OK] 전체 {len(sweep_results)}포인트 {MODEL_SCALE} 모형 일치성 확인")
        
    _speeds = sorted(set(p["speed"] for p in sweep_results))
    print(f"  속도: {_speeds} RPM, 총 {len(sweep_results)}포인트")
else:
    raise RuntimeError(f"[ERROR] JSON 파일을 찾을 수 없습니다: {json_summary_path}\n먼저 RUN_FEA_SWEEP=True로 실행하십시오.")

hybrid_data = [p for p in sweep_results if p["proximity_model"] == 1]
ts_data     = [p for p in sweep_results if p["proximity_model"] == 3]

# 매칭 진행
af_points = []
for ts_pt in ts_data:
    spd  = ts_pt["speed"]
    curr = ts_pt["current"]
    ph   = ts_pt["phase"]
    
    matches = [p for p in hybrid_data if p["speed"] == spd and np.isclose(p["current"], curr, atol=1e-2) and np.isclose(p["phase"], ph, atol=1e-2)]
    if not matches: continue
    h_pt = matches[0]
    
    h_ac = h_pt["hybrid_total_kW"]
    f_ac = ts_pt["ts_ac_active_only_kW"]
    if h_ac < 1e-4: continue
    
    af = f_ac / h_ac
    
    # dq 변환
    amp    = curr * np.sqrt(2)
    ph_rad = (ph + 90.0) * np.pi / 180.0
    id_a   = amp * np.cos(ph_rad)
    iq_a   = amp * np.sin(ph_rad)
    
    af_points.append({
        "speed_rpm":   spd,
        "speed_kRPM":  spd / 1000.0,
        "current_rms": curr,
        "phase_deg":   ph,
        "id_A":        id_a,
        "iq_A":        iq_a,
        "hybrid_ac_kW": h_ac,
        "fea_ac_kW":    f_ac,
        "AF":           af,
    })
print(f"AF 매칭 계산 완료: {len(af_points)}개 운전점")


[데이터 로드] map_exports\JEET_ACLoss_HalfSC_Map_Summary.json
  [OK] 전체 212포인트 HalfSC 모형 일치성 확인
  속도: [2000, 4000, 8000, 16000] RPM, 총 212포인트
AF 매칭 계산 완료: 106개 운전점

# [5.5] 방법 B: RBF 모델 비교 (3D TPS RBF vs. 1D x 2D 차원 분리형 RBF)

이 단계에서는 두 가지 유형의 글로벌 Thin-Plate Spline (TPS) RBF 대리 모델을 동시 수립합니다:

1. **3D TPS RBF 모델 (Full Interpolation)**:
   - 속도, 전류, 위상각 3차원 입력에 대해 106개 데이터 포인트를 모두 RBF 센터로 삼아 완벽히 매칭하는 모델입니다. 
   - 훈련 오차는 0%에 수렴하지만, Motor-CAD Lab 식의 길이가 다소 길어집니다 (~28k 캐릭터).

2. **1D x 2D 차원 분리형 스케일링 모델 (Separable Scaling Model)**:
   - 단일 속도(2.0 kRPM)의 30개 점으로 2D TPS RBF인 $g(I, \theta)$ 형상을 먼저 피팅하고, 속도 증가에 따른 스케일링 배율 $f(speed)$을 다른 속도 영역의 12개 대표 점으로 평균/2차 다항식 피팅하는 모델입니다.
   - 수식이 30개 항으로 압축되어 매우 슬림하고 (~5.5k 캐릭터), 데이터 공백에서의 과적합 없이 안정적으로 동작합니다.


In [1]:
# ─────────────────────────────────────────────────────────────────────────────
# [5.5] 방법 B: 두 가지 RBF 모델 동시 수립 (3D TPS RBF 및 1D x 2D Separable RBF)
# ─────────────────────────────────────────────────────────────────────────────
import numpy as np

# ── 데이터 준비 ──────────────────────────────────────────────────────────────
speeds_k  = np.array([p["speed_kRPM"]  for p in af_points])
irms_arr  = np.array([p["current_rms"] for p in af_points])   # Irms [A]
phase_arr = np.array([p["phase_deg"]   for p in af_points])   # phase advance [deg]
af_arr    = np.array([p["AF"]          for p in af_points])
id_arr    = np.array([p["id_A"]        for p in af_points])
iq_arr    = np.array([p["iq_A"]        for p in af_points])
curr_arr  = irms_arr.copy()
X_data    = np.column_stack([speeds_k, irms_arr, phase_arr])

# ── ARD 길이 스케일 (변수별 표준편차) ────────────────────────────────────────
LS_S = float(speeds_k.std())
LS_I = float(irms_arr.std())
LS_P = float(phase_arr.std())
print(f"  길이 스케일: ls_s={LS_S:.3f} kRPM | ls_I={LS_I:.1f} A | ls_P={LS_P:.2f} deg")

LAM = 1e-6

# ─────────────────────────────────────────────────────────────────────────────
# MODEL 1: 3D TPS RBF 모델 피팅 (106개 전체 센터)
# ─────────────────────────────────────────────────────────────────────────────
def _rbf_k_3d(s, irms, ph, s_c, i_c, p_c):
    r2 = (s - s_c)**2 / LS_S**2 + (irms - i_c)**2 / LS_I**2 + (ph - p_c)**2 / LS_P**2
    r = np.sqrt(r2)
    return r2 * np.log(r + 1e-12)

n = len(af_arr)
Phi_3d = np.zeros((n, n))
for j in range(n):
    Phi_3d[:, j] = _rbf_k_3d(speeds_k, irms_arr, phase_arr,
                             speeds_k[j], irms_arr[j], phase_arr[j])

rbf_weights_3d = np.linalg.solve(Phi_3d + LAM * np.eye(n), af_arr)

def af_from_rbf_3d(speed_rpm, irms_a, phase_deg):
    s   = np.asarray(speed_rpm, float) / 1000.0
    irm = np.asarray(irms_a,    float)
    ph  = np.asarray(phase_deg, float)
    s, irm, ph = np.broadcast_arrays(s, irm, ph)
    orig = s.shape
    sv, irmv, phv = s.ravel()[:, None], irm.ravel()[:, None], ph.ravel()[:, None]
    
    r2 = (sv - speeds_k)**2 / LS_S**2 + (irmv - irms_arr)**2 / LS_I**2 + (phv - phase_arr)**2 / LS_P**2
    r = np.sqrt(r2)
    K = r2 * np.log(r + 1e-12)
    result = K @ rbf_weights_3d
    return result.reshape(orig) if orig else float(result[0])

# ─────────────────────────────────────────────────────────────────────────────
# MODEL 2: 1D x 2D Separable RBF 모델 피팅
# ─────────────────────────────────────────────────────────────────────────────
base_idx = np.where(np.abs(speeds_k - 2.0) < 0.1)[0]
speeds_k_base = speeds_k[base_idx]
irms_arr_base = irms_arr[base_idx]
phase_arr_base = phase_arr[base_idx]
af_arr_base = af_arr[base_idx]

def _rbf_2d_k(irms, ph, i_c, p_c):
    r2 = (irms - i_c)**2 / LS_I**2 + (ph - p_c)**2 / LS_P**2
    r = np.sqrt(r2)
    return r2 * np.log(r + 1e-12)

n_base = len(base_idx)
Phi_g = np.zeros((n_base, n_base))
for j in range(n_base):
    Phi_g[:, j] = _rbf_2d_k(irms_arr_base, phase_arr_base,
                            irms_arr_base[j], phase_arr_base[j])

w_g = np.linalg.solve(Phi_g + LAM * np.eye(n_base), af_arr_base)

def predict_g(I, theta):
    I = np.asarray(I, float)
    theta = np.asarray(theta, float)
    I, theta = np.broadcast_arrays(I, theta)
    orig = I.shape
    Iv, thv = I.ravel()[:, None], theta.ravel()[:, None]
    
    r2 = (Iv - irms_arr_base)**2 / LS_I**2 + (thv - phase_arr_base)**2 / LS_P**2
    r = np.sqrt(r2)
    K = r2 * np.log(r + 1e-12)
    result = K @ w_g
    return result.reshape(orig) if orig else float(result[0])

# 1D 속도 배율 f(speed) 구하기 (4k, 8k, 16k RPM의 속도별 4점 사용)
other_speeds = [4.0, 8.0, 16.0]
target_currents = [115.0, 230.0, 345.0, 460.0]
selected_other_idx = []
for spd in other_speeds:
    spd_idx = np.where(np.abs(speeds_k - spd) < 0.1)[0]
    for i_val in target_currents:
        diffs = (irms_arr[spd_idx] - i_val)**2
        best_idx = spd_idx[np.argmin(diffs)]
        selected_other_idx.append(best_idx)
selected_other_idx = np.unique(selected_other_idx)

f_vals = []
for idx in selected_other_idx:
    spd = speeds_k[idx]
    I_val = irms_arr[idx]
    th_val = phase_arr[idx]
    af_actual = af_arr[idx]
    g_val = predict_g(I_val, th_val)
    f_val = af_actual / (g_val + 1e-12)
    f_vals.append((spd, f_val))

f_by_speed = {2.0: [1.0]}
for spd, f_val in f_vals:
    if spd not in f_by_speed:
        f_by_speed[spd] = []
    f_by_speed[spd].append(f_val)

speed_coords = []
f_coords = []
for spd in sorted(f_by_speed.keys()):
    speed_coords.append(spd)
    f_coords.append(np.mean(f_by_speed[spd]))

p_coeffs = np.polyfit(speed_coords, f_coords, 2)
p_func = np.poly1d(p_coeffs)

def af_from_rbf_separable(speed_rpm, irms_a, phase_deg):
    s = np.asarray(speed_rpm, float) / 1000.0
    irm = np.asarray(irms_a, float)
    ph = np.asarray(phase_deg, float)
    s, irm, ph = np.broadcast_arrays(s, irm, ph)
    orig = s.shape
    sv, irmv, phv = s.ravel(), irm.ravel(), ph.ravel()
    
    g_vals = predict_g(irmv, phv)
    f_vals = p_func(sv)
    result = f_vals * g_vals
    return result.reshape(orig) if orig else float(result[0])

# ── 기본 보정 함수 설정 (Separable 방식을 기본으로 사용) ──────────────────
def af_from_rbf(speed_rpm, irms_a, phase_deg):
    return af_from_rbf_separable(speed_rpm, irms_a, phase_deg)

# 다운스트림 호환용
def af_from_poly3d(speed_rpm, id_peak_a, iq_peak_a):
    idv = np.asarray(id_peak_a, float)
    iqv = np.asarray(iq_peak_a, float)
    irms  = np.sqrt(idv**2 + iqv**2) / np.sqrt(2)
    phase = np.degrees(np.arctan2(iqv, idv)) - 90.0
    return af_from_rbf(speed_rpm, irms, phase)

print("  af_from_rbf_3d() 및 af_from_rbf_separable() 수립 완료 (Separable 기본 활성화)")

# ── 3. Motor-CAD Lab 수식 포맷 ───────────────────────────────────────────────
# (1) 3D RBF 식 (106개 센터)
terms_3d = []
for j in range(n):
    w, s_c, i_c, p_c = rbf_weights_3d[j], speeds_k[j], irms_arr[j], phase_arr[j]
    r2_expr = f"((Speed/1000-{s_c:.4f})**2/{LS_S**2:.4f}+(Stator_Current_Phase_RMS-{i_c:.4f})**2/{LS_I**2:.4f}+(Phase_Advance-{p_c:.4f})**2/{LS_P**2:.4f})"
    term = f"({w:+.6f})*({r2_expr})*log({r2_expr}**0.5+1e-12)"
    terms_3d.append(term)
rbf_formula_3d = "Stator_Copper_Loss_AC * (\n  " + "\n  + ".join(terms_3d) + "\n) - Stator_Copper_Loss_AC"

# (2) Separable 식 (30개 센터)
terms_g = []
for j in range(n_base):
    w, i_c, p_c = w_g[j], irms_arr_base[j], phase_arr_base[j]
    r2_expr = f"((Stator_Current_Phase_RMS-{i_c:.4f})**2/{LS_I**2:.4f}+(Phase_Advance-{p_c:.4f})**2/{LS_P**2:.4f})"
    term = f"({w:+.6f})*({r2_expr})*log({r2_expr}**0.5+1e-12)"
    terms_g.append(term)
g_expr = " + ".join(terms_g)
f_expr = f"({p_coeffs[0]:+.6f}*(Speed/1000)**2{p_coeffs[1]:+.6f}*(Speed/1000){p_coeffs[2]:+.6f})"
rbf_formula_separable = "Stator_Copper_Loss_AC * (\n  " + f"({f_expr}) * (\n    {g_expr}\n  )" + "\n) - Stator_Copper_Loss_AC"

rbf_formula = rbf_formula_separable


길이 스케일: ls_s=5.702 kRPM | ls_I=159.3 A | ls_P=31.39 deg
  af_from_rbf_3d() 및 af_from_rbf_separable() 수립 완료 (Separable 기본 활성화)

In [1]:
# ─────────────────────────────────────────────────────────────────────────────
# [6] 방법 A: AF(speed) 속도만 2차 다항식 + AF vs Speed 시각화
# ─────────────────────────────────────────────────────────────────────────────
try:
    get_ipython().run_line_magic('matplotlib', 'inline')
except Exception:
    pass

import numpy as np
import matplotlib.pyplot as plt

# ── 방법 A 피팅: 최대 전류에서 속도만의 2차 다항식 ──────────────────────────
max_curr     = curr_arr.max()
mask_maxcurr = np.isclose(curr_arr, max_curr, rtol=0.01)
spd_mc, af_mc = speeds_k[mask_maxcurr], af_arr[mask_maxcurr]

sort_idx = np.argsort(spd_mc)
spd_mc, af_mc = spd_mc[sort_idx], af_mc[sort_idx]

coeffs_A = np.polyfit(spd_mc, af_mc, deg=2)
af_A_fit = np.polyval(coeffs_A, spd_mc)
a2, a1, a0 = coeffs_A
_coeff_A = coeffs_A.copy()

def af_from_speed_only(speed_rpm):
    return np.polyval(_coeff_A, np.asarray(speed_rpm, float) / 1000.0)

lab_formula_extra = (
    f"(({a2:.6f}*(Speed/1000)^2 + {a1:.6f}*(Speed/1000) + {a0:.6f}) - 1)"
    f" * Stator_Copper_Loss_AC"
)

print("=== 방법 A: 속도만의 2차 다항식 (최대 전류 기준) ===")
print(f"  I_rms = {max_curr:.1f} A 기준")
print(f"  AF(s) = {a2:.6f}\u00b7s\u00b2 + {a1:.6f}\u00b7s + {a0:.6f}   (s: kRPM)\n")
for s, ref, fit in zip(spd_mc, af_mc, af_A_fit):
    print(f"    {s:.0f} kRPM: AF_ref={ref:.3f}, AF_fit={fit:.3f}, \u0394={fit-ref:+.3f}")
print(f"\n  [Motor-CAD Lab \uc218\uc2dd]\n  {lab_formula_extra}")

# ── AF vs Speed 시각화 ──────────────────────────────────────────────────────
unique_currents_s = sorted(set(round(p["current_rms"], 0) for p in af_points))
unique_phases_s   = sorted(set(round(p["phase_deg"],   0) for p in af_points))
unique_speeds_s   = sorted(set(p["speed_rpm"] for p in af_points))

n_curr_s  = len(unique_currents_s)
colors_s  = [plt.cm.plasma(i / max(1, n_curr_s - 1)) for i in range(n_curr_s)]
lstyles_s = ['-', '--', '-.', ':', (0, (3, 1, 1, 1)), (0, (5, 5))]

fig, ax = plt.subplots(figsize=(11, 6))
ax.set_title("Adjustment Factor  AF = FullFEA_AC / Hybrid_AC  vs Speed (\uc6b4\uc804\uc810\ubcc4)",
             fontsize=12, fontweight='bold')

for ki, curr in enumerate(unique_currents_s):
    for li, ph in enumerate(unique_phases_s):
        pts = sorted(
            [p for p in af_points
             if np.isclose(p["current_rms"], curr, atol=0.6)
             and np.isclose(p["phase_deg"],  ph,   atol=0.6)],
            key=lambda x: x["speed_rpm"]
        )
        if len(pts) < 2:
            continue
        spds = [p["speed_rpm"] for p in pts]
        afs  = [p["AF"]        for p in pts]
        ax.plot(spds, afs,
                marker='o', markersize=5,
                linestyle=lstyles_s[li % len(lstyles_s)],
                color=colors_s[ki], linewidth=1.5,
                label=f"I={curr:.0f} A, \u03c6={ph:.0f}\u00b0")

spd_fit  = np.linspace(min(unique_speeds_s) * 0.9, max(unique_speeds_s) * 1.05, 300)
af_fit_A = af_from_speed_only(spd_fit)
eq_str = f"y = {a2:.4f}\u00b7x\u00b2 {a1:+.4f}\u00b7x {a0:+.4f}  (x: kRPM)"
ax.plot(spd_fit, af_fit_A, 'k--', linewidth=2.5,
        label=f"Poly-A fit (I_max={max_curr:.0f} A)")
ax.text(0.97, 0.97, eq_str, transform=ax.transAxes, fontsize=9,
        va='top', ha='right', bbox=dict(boxstyle='round', fc='white', alpha=0.85))

ax.axhline(y=1.0, color='green', linestyle=':', linewidth=1.5, alpha=0.7, label="AF = 1")
ax.set_xlabel("Speed [RPM]", fontsize=11)
ax.set_ylabel("Adjustment factor [-]", fontsize=11)
ax.legend(fontsize=7.5, loc='upper right', ncol=2, framealpha=0.9)
ax.grid(True, linestyle='--', alpha=0.4)
plt.tight_layout()
plt.savefig("map_exports/AF_vs_speed_curves.png", dpi=150, bbox_inches='tight')
plt.show()
print("\uc800\uc7a5: map_exports/AF_vs_speed_curves.png")


=== 방법 A: 속도만의 2차 다항식 (최대 전류 기준) ===
  I_rms = 460.0 A 기준
  AF(s) = 0.002599·s² + -0.107516·s + 2.501685   (s: kRPM)

    2 kRPM: AF_ref=3.847, AF_fit=2.297, Δ=-1.550
    2 kRPM: AF_ref=3.123, AF_fit=2.297, Δ=-0.826
    2 kRPM: AF_ref=2.502, AF_fit=2.297, Δ=-0.205
    2 kRPM: AF_ref=1.809, AF_fit=2.297, Δ=+0.488
    2 kRPM: AF_ref=1.293, AF_fit=2.297, Δ=+1.004
    2 kRPM: AF_ref=1.400, AF_fit=2.297, Δ=+0.898
    4 kRPM: AF_ref=1.216, AF_fit=2.113, Δ=+0.897
    4 kRPM: AF_ref=1.618, AF_fit=2.113, Δ=+0.495
    4 kRPM: AF_ref=1.158, AF_fit=2.113, Δ=+0.955
    4 kRPM: AF_ref=2.755, AF_fit=2.113, Δ=-0.642
    4 kRPM: AF_ref=3.382, AF_fit=2.113, Δ=-1.269
    4 kRPM: AF_ref=2.214, AF_fit=2.113, Δ=-0.101
    8 kRPM: AF_ref=1.362, AF_fit=1.808, Δ=+0.446
    8 kRPM: AF_ref=2.753, AF_fit=1.808, Δ=-0.945
    8 kRPM: AF_ref=2.254, AF_fit=1.808, Δ=-0.446
    8 kRPM: AF_ref=1.030, AF_fit=1.808, Δ=+0.778
    16 kRPM: AF_ref=2.170, AF_fit=1.447, Δ=-0.723
    16 kRPM: AF_ref=1.834, AF_fit=1.447, Δ=-0.38

<string>:84: UserWarning: Glyph 50868 (\N{HANGUL SYLLABLE UN}) missing from current font.
<string>:84: UserWarning: Glyph 51204 (\N{HANGUL SYLLABLE JEON}) missing from current font.
<string>:84: UserWarning: Glyph 51216 (\N{HANGUL SYLLABLE JEOM}) missing from current font.
<string>:84: UserWarning: Glyph 48324 (\N{HANGUL SYLLABLE BYEOL}) missing from current font.
<string>:85: UserWarning: Glyph 50868 (\N{HANGUL SYLLABLE UN}) missing from current font.
<string>:85: UserWarning: Glyph 51204 (\N{HANGUL SYLLABLE JEON}) missing from current font.
<string>:85: UserWarning: Glyph 51216 (\N{HANGUL SYLLABLE JEOM}) missing from current font.
<string>:85: UserWarning: Glyph 48324 (\N{HANGUL SYLLABLE BYEOL}) missing from current font.

# [6.5] 방법 B 시각화: id-iq 평면 AF 분포 (분리형 RBF)

속도별 id-iq 평면에서의 AF 예측 거동 및 분리형 RBF 곡면을 시각화합니다.


In [1]:
# ─────────────────────────────────────────────────────────────────────────────
# [6.5] 방법 B 시각화: id-iq 평면 AF 맵 (속도별)
# ─────────────────────────────────────────────────────────────────────────────
try:
    get_ipython().run_line_magic('matplotlib', 'inline')
except Exception:
    pass

import numpy as np
import matplotlib.pyplot as plt

unique_speeds_v = sorted(set(p["speed_rpm"] for p in af_points))
n_spd_v = len(unique_speeds_v)

fig, axes = plt.subplots(1, n_spd_v, figsize=(5.2 * n_spd_v, 4.8))
if n_spd_v == 1:
    axes = [axes]
fig.suptitle("Adjustment Factor  AF = FullFEA_AC / Hybrid_AC  (id-iq 평면)",
             fontsize=13, fontweight='bold')

af_vals_all = np.array([p["AF"] for p in af_points])
vmin_af = max(0.5, af_vals_all.min() - 0.1)
vmax_af = af_vals_all.max() + 0.1

for ax, spd in zip(axes, unique_speeds_v):
    pts  = [p for p in af_points if p["speed_rpm"] == spd]
    id_v = np.array([p["id_A"] for p in pts])
    iq_v = np.array([p["iq_A"] for p in pts])
    af_v = np.array([p["AF"]   for p in pts])

    sc = ax.scatter(id_v, iq_v, c=af_v, cmap='plasma', s=90,
                    edgecolors='k', linewidths=0.6,
                    vmin=vmin_af, vmax=vmax_af, zorder=3)
    for x, y, a in zip(id_v, iq_v, af_v):
        ax.annotate(f"{a:.2f}", (x, y), textcoords="offset points",
                    xytext=(5, 4), fontsize=7.5, color='black')

    pad = 80
    id_g = np.linspace(id_v.min() - pad, id_v.max() + pad, 50)
    iq_g = np.linspace(max(0, iq_v.min() - pad), iq_v.max() + pad, 50)
    ID, IQ = np.meshgrid(id_g, iq_g)
    AF_fit = af_from_poly3d(spd, ID.ravel(), IQ.ravel()).reshape(ID.shape)
    ct = ax.contour(ID, IQ, AF_fit, levels=8, cmap='coolwarm', alpha=0.65, linewidths=0.9)
    ax.clabel(ct, fmt="%.2f", fontsize=7.5)

    plt.colorbar(sc, ax=ax, label="AF [-]", shrink=0.85)
    ax.set_xlabel("$I_d$ [A, peak]", fontsize=9)
    ax.set_ylabel("$I_q$ [A, peak]", fontsize=9)
    ax.set_title(f"{spd/1000:.0f} kRPM", fontsize=11, fontweight='bold')
    ax.grid(True, linestyle='--', alpha=0.4)

plt.tight_layout()
plt.savefig("map_exports/AF_map_visualization.png", dpi=150, bbox_inches='tight')
plt.show()
print("저장 완료: map_exports/AF_map_visualization.png")


저장 완료: map_exports/AF_map_visualization.png

<string>:52: UserWarning: Glyph 54217 (\N{HANGUL SYLLABLE PYEONG}) missing from current font.
<string>:52: UserWarning: Glyph 47732 (\N{HANGUL SYLLABLE MYEON}) missing from current font.
<string>:53: UserWarning: Glyph 54217 (\N{HANGUL SYLLABLE PYEONG}) missing from current font.
<string>:53: UserWarning: Glyph 47732 (\N{HANGUL SYLLABLE MYEON}) missing from current font.

In [1]:
# ─────────────────────────────────────────────────────────────────────────────
# [6.6] 방법 B 3D 표면 시각화: AF(id, iq) 곡면 (속도별)
# ─────────────────────────────────────────────────────────────────────────────
try:
    get_ipython().run_line_magic("matplotlib", "inline")
except Exception:
    pass

import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401

unique_speeds_v = sorted(set(p["speed_rpm"] for p in af_points))
n_spd_v = len(unique_speeds_v)

fig = plt.figure(figsize=(5.5 * n_spd_v, 5.0))
fig.suptitle("AF Surface: AF(Id, Iq) 방법 B 3D 곡면 (속도별)", fontsize=13, fontweight="bold")

for k, spd in enumerate(unique_speeds_v):
    ax = fig.add_subplot(1, n_spd_v, k + 1, projection="3d")

    pts  = [p for p in af_points if p["speed_rpm"] == spd]
    id_v = np.array([p["id_A"] for p in pts])
    iq_v = np.array([p["iq_A"] for p in pts])
    af_v = np.array([p["AF"]   for p in pts])

    pad = 80
    id_g = np.linspace(id_v.min() - pad, id_v.max() + pad, 50)
    iq_g = np.linspace(max(0, iq_v.min() - pad), iq_v.max() + pad, 50)
    ID, IQ = np.meshgrid(id_g, iq_g)
    AF_fit = af_from_poly3d(spd, ID, IQ)

    surf = ax.plot_surface(ID, IQ, AF_fit, cmap="plasma", alpha=0.75,
                           linewidth=0, antialiased=True)
    ax.scatter(id_v, iq_v, af_v, c="red", s=60,
               edgecolors="k", linewidths=0.6, zorder=5, label="FEA data")

    fig.colorbar(surf, ax=ax, shrink=0.55, label="AF [-]")
    ax.set_xlabel("Id [A]", fontsize=8)
    ax.set_ylabel("Iq [A]", fontsize=8)
    ax.set_zlabel("AF [-]", fontsize=8)
    ax.set_title(f"{spd/1000:.0f} kRPM", fontsize=11, fontweight="bold")
    ax.view_init(elev=25, azim=-60)

plt.tight_layout()
plt.savefig("map_exports/AF_3D_surface.png", dpi=150, bbox_inches="tight")
plt.show()
print("저장 완료: map_exports/AF_3D_surface.png")


저장 완료: map_exports/AF_3D_surface.png

<string>:45: UserWarning: Glyph 48169 (\N{HANGUL SYLLABLE BANG}) missing from current font.
<string>:45: UserWarning: Glyph 48277 (\N{HANGUL SYLLABLE BEOB}) missing from current font.
<string>:45: UserWarning: Glyph 44257 (\N{HANGUL SYLLABLE GOG}) missing from current font.
<string>:45: UserWarning: Glyph 47732 (\N{HANGUL SYLLABLE MYEON}) missing from current font.
<string>:45: UserWarning: Glyph 49549 (\N{HANGUL SYLLABLE SOG}) missing from current font.
<string>:45: UserWarning: Glyph 46020 (\N{HANGUL SYLLABLE DO}) missing from current font.
<string>:45: UserWarning: Glyph 48324 (\N{HANGUL SYLLABLE BYEOL}) missing from current font.
<string>:46: UserWarning: Glyph 48169 (\N{HANGUL SYLLABLE BANG}) missing from current font.
<string>:46: UserWarning: Glyph 48277 (\N{HANGUL SYLLABLE BEOB}) missing from current font.
<string>:46: UserWarning: Glyph 44257 (\N{HANGUL SYLLABLE GOG}) missing from current font.
<string>:46: UserWarning: Glyph 47732 (\N{HANGUL SYLLABLE MYEON}) missing from curr

# [7] 대리 모델 성능 비교 및 시각화

- 3D TPS RBF 모델과 1D x 2D Separable RBF 모델의 예측 오차(Train MAE 및 Leave-One-Out CV 오차)를 직접 비교합니다.
- 두 모델의 예측 데이터 Parity Plot 및 3-way Boxplot을 생성하여 비교 시각화하고 최종 JSON 데이터를 내보냅니다.


In [1]:
# ─────────────────────────────────────────────────────────────────────────────
# [7] RBF 모델 비교 검증: 3D TPS RBF vs. 1D x 2D Separable RBF vs. FullFEA
# ─────────────────────────────────────────────────────────────────────────────
try:
    get_ipython().run_line_magic('matplotlib', 'inline')
except Exception:
    pass

import numpy as np
import matplotlib.pyplot as plt
import json
from pathlib import Path

print(f"=== RBF 보정 오차 검증 및 비교: 3D RBF vs Separable vs FullFEA ({MODEL_SCALE}) ===\n")

h_ac_arr = np.array([p["hybrid_ac_kW"] for p in af_points])
f_ac_arr = np.array([p["fea_ac_kW"] for p in af_points])

# ── 훈련 세트 오차 계산
err_raw, err_3d, err_sep = [], [], []
rows = []
for p in af_points:
    spd   = p["speed_rpm"]
    irms  = p["current_rms"]
    phase = p["phase_deg"]
    h_ac  = p["hybrid_ac_kW"]
    f_ac  = p["fea_ac_kW"]
    
    af_3d  = float(af_from_rbf_3d(spd, irms, phase))
    af_sep = float(af_from_rbf_separable(spd, irms, phase))
    
    corr_3d  = h_ac * af_3d
    corr_sep = h_ac * af_sep
    
    e_raw = (h_ac - f_ac) / (f_ac + 1e-12) * 100
    e_3d  = (corr_3d - f_ac) / (f_ac + 1e-12) * 100
    e_sep = (corr_sep - f_ac) / (f_ac + 1e-12) * 100
    
    err_raw.append(e_raw)
    err_3d.append(e_3d)
    err_sep.append(e_sep)
    rows.append((spd, irms, phase, h_ac, f_ac, corr_3d, corr_sep, e_raw, e_3d, e_sep))

ea = np.array(err_raw)
e3 = np.array(err_3d)
es = np.array(err_sep)

# ── LOOCV 계산
print("  LOOCV 계산 중 (약 1.5초 소요)... ")
loocv_errors_3d = []
for i in range(n):
    X_tr = np.delete(X_data, i, axis=0)
    y_tr = np.delete(af_arr, i, axis=0)
    Phi_tr = np.zeros((n-1, n-1))
    for j in range(n-1):
        Phi_tr[:, j] = _rbf_k_3d(X_tr[:, 0], X_tr[:, 1], X_tr[:, 2],
                                 X_tr[j, 0], X_tr[j, 1], X_tr[j, 2])
    w_tr = np.linalg.solve(Phi_tr + LAM * np.eye(n-1), y_tr)
    
    r2 = (X_data[i, 0] - X_tr[:, 0])**2 / LS_S**2        + (X_data[i, 1] - X_tr[:, 1])**2 / LS_I**2        + (X_data[i, 2] - X_tr[:, 2])**2 / LS_P**2
    r = np.sqrt(r2)
    K = r2 * np.log(r + 1e-12)
    y_pred = K @ w_tr
    corr_val = h_ac_arr[i] * y_pred
    loocv_errors_3d.append(abs((corr_val - f_ac_arr[i]) / f_ac_arr[i] * 100))
mae_loocv_3d = np.mean(loocv_errors_3d)

loocv_errors_sep = []
for i in range(n):
    base_train_idx = [idx for idx in base_idx if idx != i]
    X_base_tr = X_data[base_train_idx, 1:3]
    y_base_tr = af_arr[base_train_idx]
    
    Phi_g_tr = np.zeros((len(base_train_idx), len(base_train_idx)))
    for j in range(len(base_train_idx)):
        Phi_g_tr[:, j] = _rbf_2d_k(X_base_tr[:, 0], X_base_tr[:, 1],
                                    X_base_tr[j, 0], X_base_tr[j, 1])
    w_g_tr = np.linalg.solve(Phi_g_tr + LAM * np.eye(len(base_train_idx)), y_base_tr)
    
    def predict_g_tr(I, theta):
        I = np.asarray(I, float)
        theta = np.asarray(theta, float)
        I, theta = np.broadcast_arrays(I, theta)
        orig = I.shape

        Iv, thv = I.ravel()[:, None], theta.ravel()[:, None]
        r2 = (Iv - X_base_tr[:, 0])**2 / LS_I**2 + (thv - X_base_tr[:, 1])**2 / LS_P**2
        r = np.sqrt(r2)
        K = r2 * np.log(r + 1e-12)
        result = K @ w_g_tr
        return result.reshape(orig) if orig else float(result[0])
        
    cal_train_idx = [idx for idx in selected_other_idx if idx != i]
    f_vals_tr = []
    for idx in cal_train_idx:
        spd = speeds_k[idx]
        I_val = irms_arr[idx]
        th_val = phase_arr[idx]
        af_actual = af_arr[idx]
        g_val = predict_g_tr(I_val, th_val)
        f_val = af_actual / (g_val + 1e-12)
        f_vals_tr.append((spd, f_val))
        
    f_by_speed_tr = {2.0: [1.0]}
    for spd, f_val in f_vals_tr:
        if spd not in f_by_speed_tr:
            f_by_speed_tr[spd] = []
        f_by_speed_tr[spd].append(f_val)
        
    speed_coords_tr = []
    f_coords_tr = []
    for spd in sorted(f_by_speed_tr.keys()):
        speed_coords_tr.append(spd)
        f_coords_tr.append(np.mean(f_by_speed_tr[spd]))
        
    p_coeffs_tr = np.polyfit(speed_coords_tr, f_coords_tr, 2)
    p_func_tr = np.poly1d(p_coeffs_tr)
    
    g_val_i = predict_g_tr(X_data[i, 1], X_data[i, 2])
    f_val_i = p_func_tr(X_data[i, 0])
    y_pred_i = f_val_i * g_val_i
    corr_val = h_ac_arr[i] * y_pred_i
    loocv_errors_sep.append(abs((corr_val - f_ac_arr[i]) / f_ac_arr[i] * 100))
mae_loocv_sep = np.mean(loocv_errors_sep)

print("=== RBF 보정 오차 최종 비교 결과 ===")
print(f"  1) Hybrid (보정 전):        Train MAE={np.abs(ea).mean():.2f}% | MaxAE={np.abs(ea).max():.2f}%")
print(f"  2) 3D TPS RBF:             Train MAE={np.abs(e3).mean():.2f}% | MaxAE={np.abs(e3).max():.2f}% | LOOCV MAE={mae_loocv_3d:.2f}%")
print(f"  3) Separable (분리형 RBF):   Train MAE={np.abs(es).mean():.2f}% | MaxAE={np.abs(es).max():.2f}% | LOOCV MAE={mae_loocv_sep:.2f}%")

# ── 시각화 및 그림 저장
fea_all   = np.array([r[4] for r in rows])
hybr_all  = np.array([r[3] for r in rows])
corr_3d   = np.array([r[5] for r in rows])
corr_sep  = np.array([r[6] for r in rows])

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle(f"RBF Model Comparison ({MODEL_SCALE}): 3D RBF vs Separable vs FullFEA", fontsize=12, fontweight='bold')

ax = axes[0]
lim = [min(fea_all.min(), hybr_all.min(), corr_3d.min(), corr_sep.min()) * 0.9,
       max(fea_all.max(), hybr_all.max(), corr_3d.max(), corr_sep.max()) * 1.05]
ax.plot(lim, lim, 'k--', linewidth=1.2, label='Perfect fit')
ax.scatter(fea_all, hybr_all, c='grey',      s=30, alpha=0.5, label='Hybrid (보정 전)', zorder=2)
ax.scatter(fea_all, corr_3d,  c='steelblue', s=45, alpha=0.7, label=f'3D RBF (LOOCV: {mae_loocv_3d:.2f}%)', zorder=3)
ax.scatter(fea_all, corr_sep, c='tomato',    s=45, alpha=0.8, label=f'Separable (LOOCV: {mae_loocv_sep:.2f}%)', zorder=4)
ax.set_xlabel("FullFEA AC Loss [kW]", fontsize=10)
ax.set_ylabel("Predicted AC Loss [kW]", fontsize=10)
ax.set_title("Parity Plot", fontsize=11)
ax.legend(fontsize=9); ax.grid(True, linestyle='--', alpha=0.4)
ax.set_xlim(lim); ax.set_ylim(lim)

ax2 = axes[1]
bp = ax2.boxplot([ea, e3, es], labels=['Hybrid (보정 전)', '3D RBF', 'Separable RBF'], patch_artist=True, widths=0.4)
bp['boxes'][0].set_facecolor('grey');       bp['boxes'][0].set_alpha(0.4)
bp['boxes'][1].set_facecolor('steelblue');  bp['boxes'][1].set_alpha(0.6)
bp['boxes'][2].set_facecolor('tomato');     bp['boxes'][2].set_alpha(0.6)
ax2.axhline(0, color='k', linestyle='--', linewidth=1)
ax2.set_ylabel("오차 [%]", fontsize=10)
ax2.set_title("Error Distribution Comparison", fontsize=11)
ax2.grid(True, linestyle='--', alpha=0.4)
for i, (arr, x) in enumerate([(ea, 1), (e3, 2), (es, 3)]):
    ax2.text(x, arr.max() + 0.5, f"MAE={np.abs(arr).mean():.1f}%", ha='center', fontsize=8.5, color='black')

plt.tight_layout()
val_plot_path = out_dir / f"RBF_correction_validation_{MODEL_SCALE}.png"
plt.savefig(val_plot_path, dpi=150, bbox_inches='tight')
plt.show()
print(f"그림 저장 완료: {val_plot_path}")

# ── 모델 내보내기
export = {
    "model_type": f"RBF_{MODEL_SCALE}",
    "3D_model": {
        "model": "3D_TPS_RBF",
        "n_centers": int(n),
        "weights": rbf_weights_3d.tolist(),
        "validation": {
            "Train_MAE_pct": float(np.abs(e3).mean()),
            "LOOCV_MAE_pct": float(mae_loocv_3d),
        },
        "mcad_formula": rbf_formula_3d
    },
    "separable_model": {
        "model": "Separable_1D_2D_RBF",
        "n_base_centers": int(n_base),
        "base_weights": w_g.tolist(),
        "speed_poly_coeffs": p_coeffs.tolist(),
        "validation": {
            "Train_MAE_pct": float(np.abs(es).mean()),
            "LOOCV_MAE_pct": float(mae_loocv_sep),
        },
        "mcad_formula": rbf_formula_separable
    },
    "mcad_formula_full": rbf_formula_3d,
    "mcad_formula_reduced_30": rbf_formula_separable,
    "mcad_formula_top20": rbf_formula_separable,
    "length_scales": {"LS_S_kRPM": float(LS_S), "LS_I_A": float(LS_I), "LS_P_deg": float(LS_P)},
    "af_points": af_points
}
with open(rbf_model_path, "w", encoding="utf-8") as f:
    json.dump(export, f, ensure_ascii=False, indent=2)
print(f"JSON 모델 저장 완료: {rbf_model_path}")

# ── [대화형 4-Way 비교 플롯 구현] ───────────────────────────────────────────
try:
    import IPython
    shell = IPython.get_ipython()
    if shell is not None:
        import os, sys
        has_vscode_env = any(k.startswith('VSCODE_') for k in os.environ.keys())
        has_vscode_modules = any('vscode' in m.lower() for m in sys.modules.keys())
        selected_backend = 'widget' if (has_vscode_env and has_vscode_modules) else 'inline'
        if selected_backend == 'widget':
            shell.run_line_magic('matplotlib', 'widget')
        else:
            shell.run_line_magic('matplotlib', 'inline')
except Exception:
    pass

import matplotlib.patches as mpatches
from mpl_toolkits.mplot3d import Axes3D

print("\n  대화형 4-Way 비교 플롯 로딩...")
speeds = np.array([p["speed_rpm"] for p in af_points])
irms = np.array([p["current_rms"] for p in af_points])
phases = np.array([p["phase_deg"] for p in af_points])
id_vals = np.array([p["id_A"] for p in af_points])
iq_vals = np.array([p["iq_A"] for p in af_points])

loss_hyb = np.array([p["hybrid_ac_kW"] for p in af_points])
loss_fea = np.array([p["fea_ac_kW"] for p in af_points])
loss_3d  = np.array([float(af_from_rbf_3d(p["speed_rpm"], p["current_rms"], p["phase_deg"])) * p["hybrid_ac_kW"] for p in af_points])
loss_sep = np.array([float(af_from_rbf_separable(p["speed_rpm"], p["current_rms"], p["phase_deg"])) * p["hybrid_ac_kW"] for p in af_points])

fig_int = plt.figure(figsize=(17, 8.5))
fig_int.suptitle(f"AC Loss 3D Map & Speed Curve Comparison ({MODEL_SCALE}): Hybrid vs 3D RBF vs Separable RBF vs FullFEA", fontsize=13, fontweight='bold')

ax_hyb = fig_int.add_subplot(2, 3, 1, projection='3d')
ax_3d  = fig_int.add_subplot(2, 3, 2, projection='3d')
ax_sep = fig_int.add_subplot(2, 3, 4, projection='3d')
ax_fea = fig_int.add_subplot(2, 3, 5, projection='3d')
ax_curve = fig_int.add_subplot(2, 3, (3, 6))

ax_hyb.set_title("1) Hybrid (보정 전)", fontsize=11, fontweight='bold')
ax_3d.set_title("2) 3D TPS RBF (보정 후)", fontsize=11, fontweight='bold')
ax_sep.set_title("3) Separable RBF (보정 후)", fontsize=11, fontweight='bold')
ax_fea.set_title("4) FullFEA (참조값)", fontsize=11, fontweight='bold')

unique_speeds = sorted(list(set(speeds)))
speed_colors = {2000: 'cyan', 4000: 'limegreen', 8000: 'orange', 16000: 'tomato'}
default_colors = ['cyan', 'limegreen', 'orange', 'tomato']
axes_3d = [ax_hyb, ax_3d, ax_sep, ax_fea]
losses_list = [loss_hyb, loss_3d, loss_sep, loss_fea]

legend_patches = []
for i, spd in enumerate(unique_speeds):
    color = speed_colors.get(spd, default_colors[i % len(default_colors)])
    legend_patches.append(mpatches.Patch(color=color, alpha=0.35, label=f"{spd} RPM"))
    idx_spd = (speeds == spd)
    if np.any(idx_spd) and np.sum(idx_spd) >= 3:
        for ax, loss_val in zip(axes_3d, losses_list):
            ax.plot_trisurf(id_vals[idx_spd], iq_vals[idx_spd], loss_val[idx_spd], color=color, edgecolor='none', alpha=0.2)

sc_hyb = ax_hyb.scatter(id_vals, iq_vals, loss_hyb, c='grey', s=20, picker=True, pickradius=5, edgecolors='black', alpha=0.6)
sc_3d  = ax_3d.scatter(id_vals, iq_vals, loss_3d,   c='grey', s=20, picker=True, pickradius=5, edgecolors='black', alpha=0.6)
sc_sep = ax_sep.scatter(id_vals, iq_vals, loss_sep, c='grey', s=20, picker=True, pickradius=5, edgecolors='black', alpha=0.6)
sc_fea = ax_fea.scatter(id_vals, iq_vals, loss_fea, c='grey', s=20, picker=True, pickradius=5, edgecolors='black', alpha=0.6)
scatters = [sc_hyb, sc_3d, sc_sep, sc_fea]

for ax in axes_3d:
    ax.set_xlabel("I_d [A]", fontsize=8, labelpad=5)
    ax.set_ylabel("I_q [A]", fontsize=8, labelpad=5)
    ax.set_zlabel("AC Loss [kW]", fontsize=8, labelpad=5)
    ax.legend(handles=legend_patches, fontsize=8)

ax_curve.text(0.5, 0.5, "3D 플롯에서 임의의 점을 클릭한 후\nSpacebar를 누르거나 클릭하면 우측에 속도별 비교 곡선이 출력됩니다.", 
             ha="center", va="center", fontsize=10, color="gray")
ax_curve.set_xlabel("Speed [RPM]", fontsize=9)
ax_curve.set_ylabel("AC Loss [kW]", fontsize=9)
ax_curve.grid(True, linestyle="--", alpha=0.5)

selected_pt = {"current_rms": None, "phase_deg": None, "id_A": None, "iq_A": None}
highlights = []
annots = []
for ax in axes_3d:
    annot = ax.text2D(0.02, 0.95, "", transform=ax.transAxes, bbox=dict(boxstyle="round", fc="w", alpha=0.8), fontsize=8)
    annot.set_visible(False)
    annots.append(annot)

def update_2d_curve(curr, ph):
    ax_curve.clear()
    match_pts = [p for p in af_points if np.isclose(p["current_rms"], curr) and np.isclose(p["phase_deg"], ph)]
    match_pts = sorted(match_pts, key=lambda x: x["speed_rpm"])
    
    curve_speeds = [p["speed_rpm"] for p in match_pts]
    c_loss_hyb = [p["hybrid_ac_kW"] for p in match_pts]
    c_loss_fea = [p["fea_ac_kW"] for p in match_pts]
    c_loss_3d  = [float(af_from_rbf_3d(p["speed_rpm"], p["current_rms"], p["phase_deg"])) * p["hybrid_ac_kW"] for p in match_pts]
    c_loss_sep = [float(af_from_rbf_separable(p["speed_rpm"], p["current_rms"], p["phase_deg"])) * p["hybrid_ac_kW"] for p in match_pts]
    
    ax_curve.plot(curve_speeds, c_loss_hyb, marker='o', linestyle='-',  color='grey',      linewidth=1.5, label="1) Hybrid (보정 전)")
    ax_curve.plot(curve_speeds, c_loss_3d,  marker='s', linestyle='-',  color='steelblue', linewidth=2,   label="2) 3D RBF (보정 후)")
    ax_curve.plot(curve_speeds, c_loss_sep, marker='^', linestyle='-',  color='tomato',    linewidth=2,   label="3) Separable (보정 후)")
    ax_curve.plot(curve_speeds, c_loss_fea, marker='*', linestyle='--', color='black',     linewidth=2,   label="4) FullFEA Reference")
    
    for xs, yh, y3, ys, yf in zip(curve_speeds, c_loss_hyb, c_loss_3d, c_loss_sep, c_loss_fea):
        ax_curve.annotate(f"{yh:.2f}", xy=(xs, yh), xytext=(4, 8),   textcoords="offset points", fontsize=8, color="grey")
        ax_curve.annotate(f"{y3:.2f}", xy=(xs, y3), xytext=(4, 0),   textcoords="offset points", fontsize=8, color="steelblue")
        ax_curve.annotate(f"{ys:.2f}", xy=(xs, ys), xytext=(4, -8),  textcoords="offset points", fontsize=8, color="tomato")
        ax_curve.annotate(f"{yf:.2f}", xy=(xs, yf), xytext=(4, -16), textcoords="offset points", fontsize=8, color="black")
        
    ax_curve.set_title(f"AC Loss vs Speed Comparison\n(I_rms={curr:.1f}A, Phase={ph:.1f}°)", fontsize=11, fontweight='bold')
    ax_curve.set_xlabel("Speed [RPM]", fontsize=9)
    ax_curve.set_ylabel("AC Loss [kW]", fontsize=9)
    ax_curve.grid(True, linestyle="--", alpha=0.5)
    ax_curve.legend(fontsize=9, loc="upper left")

def on_pick(event):
    if event.artist not in scatters:
        return
    idx = event.ind[0]
    p_sel = af_points[idx]
    curr = p_sel["current_rms"]
    ph = p_sel["phase_deg"]
    
    selected_pt["current_rms"] = curr
    selected_pt["phase_deg"] = ph
    selected_pt["id_A"] = p_sel["id_A"]
    selected_pt["iq_A"] = p_sel["iq_A"]
    
    for h in highlights:
        h.remove()
    highlights.clear()
    
    same_pt_idx = np.where((irms == curr) & (phases == ph))[0]
    for ax, loss_val in zip(axes_3d, losses_list):
        h = ax.scatter(id_vals[same_pt_idx], iq_vals[same_pt_idx], loss_val[same_pt_idx], color='red', s=60, edgecolors='black', linewidths=1.5, zorder=10)
        highlights.append(h)
        
    msg = f"Selected: I_rms={curr:.1f}A, Phase={ph:.1f}°\nId={selected_pt['id_A']:.1f}A, Iq={selected_pt['iq_A']:.1f}A"
    for annot in annots:
        annot.set_text(msg)
        annot.set_visible(True)
        
    update_2d_curve(curr, ph)
    fig_int.canvas.draw_idle()

def on_key(event):
    if event.key != ' ' or selected_pt["current_rms"] is None:
        return
    update_2d_curve(selected_pt["current_rms"], selected_pt["phase_deg"])
    fig_int.canvas.draw_idle()

fig_int.canvas.mpl_connect('pick_event', on_pick)
fig_int.canvas.mpl_connect('key_press_event', on_key)
plt.tight_layout()
plt.show()


=== RBF 보정 오차 검증 및 비교: 3D RBF vs Separable vs FullFEA (HalfSC) ===

  LOOCV 계산 중 (약 1.5초 소요)... 
=== RBF 보정 오차 최종 비교 결과 ===
  1) Hybrid (보정 전):        Train MAE=39.70% | MaxAE=78.23%
  2) 3D TPS RBF:             Train MAE=0.00% | MaxAE=0.00% | LOOCV MAE=4.95%
  3) Separable (분리형 RBF):   Train MAE=5.02% | MaxAE=40.69% | LOOCV MAE=8.49%
그림 저장 완료: map_exports\RBF_correction_validation_HalfSC.png
JSON 모델 저장 완료: map_exports\AF_RBF_model_HalfSC.json

  대화형 4-Way 비교 플롯 로딩...

<string>:165: UserWarning: Glyph 48372 (\N{HANGUL SYLLABLE BO}) missing from current font.
<string>:165: UserWarning: Glyph 51221 (\N{HANGUL SYLLABLE JEONG}) missing from current font.
<string>:165: UserWarning: Glyph 51204 (\N{HANGUL SYLLABLE JEON}) missing from current font.
<string>:165: UserWarning: Glyph 50724 (\N{HANGUL SYLLABLE O}) missing from current font.
<string>:165: UserWarning: Glyph 52264 (\N{HANGUL SYLLABLE CA}) missing from current font.
<string>:167: UserWarning: Glyph 48372 (\N{HANGUL SYLLABLE BO}) missing from current font.
<string>:167: UserWarning: Glyph 51221 (\N{HANGUL SYLLABLE JEONG}) missing from current font.
<string>:167: UserWarning: Glyph 51204 (\N{HANGUL SYLLABLE JEON}) missing from current font.
<string>:167: UserWarning: Glyph 50724 (\N{HANGUL SYLLABLE O}) missing from current font.
<string>:167: UserWarning: Glyph 52264 (\N{HANGUL SYLLABLE CA}) missing from current font.
<string>:357: UserWarning: Glyph 48372 (\N{HANGUL SYLLABLE BO}) missing from curre